# Quadragon v1 - PPO Training (Colab)

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine).

Assumes your Drive has a folder `quadragon_v1/` containing:
`quadragon_env.py`, `train.py`, `smoke_test.py`, `quadragon_mg995.xml`, and the 13 STLs next to the xml.

**Key design decision in this notebook:** checkpoints save directly to Drive, not local disk — Colab kills sessions without warning, and anything on local disk dies with it. A 12h run that checkpoints locally can lose everything at 11h59m.

## 1. Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT if your Drive folder lives elsewhere
DRIVE_PROJECT = '/content/drive/MyDrive/quadragon_v1'

import os
assert os.path.exists(f'{DRIVE_PROJECT}/quadragon_mg995.xml'), f'Model xml not found in {DRIVE_PROJECT} - check the path'
print('Drive folder found:', os.listdir(DRIVE_PROJECT))

## 2. Copy project to local disk
Training reads the model xml + meshes constantly — running from Drive directly is slow and flaky. Code and meshes go local; only checkpoints go back to Drive.

In [ ]:
!rm -rf /content/quadragon_v1
!cp -r "$DRIVE_PROJECT" /content/quadragon_v1
%cd /content/quadragon_v1
!ls

## 3. Install dependencies

In [ ]:
!pip install -q mujoco gymnasium stable-baselines3 tensorboard

# Headless rendering backend for rollout videos later (Colab GPU runtimes support EGL)
import os
os.environ['MUJOCO_GL'] = 'egl'

import mujoco, torch
print('MuJoCo:', mujoco.__version__)
print('CUDA available:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 4. Smoke test
Same five checks that passed in development — run once per fresh session to confirm the transfer to Colab is clean before burning GPU time.

In [ ]:
!python3 smoke_test.py

## 5. TensorBoard (start before training so it's live during the run)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 6. Train
Checkpoints write straight to Drive every ~100k steps via symlink, so a dead session costs you at most the steps since the last checkpoint.

PPO itself is a small MLP — the env stepping (CPU) is the bottleneck, not the GPU, so `n_envs` matters more than GPU model here. 8 is right for Colab's 2-core CPU; on the 4090 machine (more cores) push it to 16-32.

In [ ]:
# Checkpoints -> Drive (survives disconnects)
DRIVE_CKPT = f'{DRIVE_PROJECT}/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
!rm -rf checkpoints && ln -s "$DRIVE_CKPT" checkpoints

!python3 train.py --n-envs 8 --timesteps 3000000 --run-name v1_colab

## 7. Save final model + normalization stats to Drive
Both files. The policy is meaningless without its matching VecNormalize stats.

In [ ]:
!cp quadragon_v1_colab_final.zip vecnormalize_v1_colab.pkl "$DRIVE_PROJECT/" 2>/dev/null && echo 'Saved to Drive' || echo 'Final files not found - if the session died mid-run, grab the latest from checkpoints/ instead'

## 8. Watch it - render a rollout video
The reward curve tells you *if* it's learning; the video tells you *what* it learned. A rising curve can still be reward hacking (vibrating forward, dragging its belly) — always watch before trusting.

In [ ]:
import numpy as np
import imageio
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from quadragon_env import QuadragonEnv

MODEL_PATH = 'quadragon_v1_colab_final.zip'      # or a checkpoints/ file
VECNORM_PATH = 'vecnormalize_v1_colab.pkl'

venv = DummyVecEnv([lambda: QuadragonEnv(render_mode='rgb_array')])
venv = VecNormalize.load(VECNORM_PATH, venv)
venv.training = False        # freeze stats - deployment-identical inference
venv.norm_reward = False

model = PPO.load(MODEL_PATH)
raw_env = venv.venv.envs[0]

obs = venv.reset()
frames, total_r = [], 0.0
for step in range(500):   # 10s at 50Hz
    action, _ = model.predict(obs, deterministic=True)
    obs, r, done, info = venv.step(action)
    total_r += r[0]
    if step % 2 == 0:      # 25fps video
        frames.append(raw_env.render())
    if done[0]:
        print(f'Episode ended at step {step}')
        break

print(f'Total reward: {total_r:.1f}, final vx: {info[0]["vx"]:.3f} m/s')
imageio.mimsave('rollout.mp4', frames, fps=25)

from IPython.display import Video
Video('rollout.mp4', embed=True, width=640)

## Resuming after a disconnect
Re-run cells 1-5, then instead of cell 6's fresh `train.py`, load the latest checkpoint:
```python
# find newest checkpoint
import glob
latest = sorted(glob.glob('checkpoints/quadragon_v1_colab_*_steps.zip'))[-1]
print('Resuming from', latest)
```
then in Python: `model = PPO.load(latest, env=venv)` + `model.learn(remaining_steps, reset_num_timesteps=False)` — or just start a fresh run if you were early in training; before ~500k steps a restart costs little.